# Assessment 2: Concepts, Splits and Leakage

## Student Performance Dataset

### Objectives
1. Understand machine learning concepts.
2. Identify independent and dependent variables.
3. Perform training and testing data splits.
4. Understand data leakage.
5. Prevent data leakage using StandardScaler.

---

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Locate project directory
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "student_performance.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("STUDENT PERFORMANCE DATASET")
display(df)

print("\nDataset Shape:", df.shape)

STUDENT PERFORMANCE DATASET


,Hours_Studied,Attendance,Previous_Score,Final_Score
0,2,60,40,45
1,3,65,45,50
2,4,70,50,55
3,5,75,55,60
4,6,80,60,65
5,7,85,65,70
6,8,90,70,75
7,9,95,75,80
8,10,98,80,85
9,1,55,35,40



Dataset Shape: (10, 4)


In [4]:
# Independent variables (Features)
X = df[["Hours_Studied", "Attendance", "Previous_Score"]]

# Dependent variable (Target)
y = df["Final_Score"]

print("FEATURES:")
display(X)

print("\nTARGET:")
display(y)

FEATURES:


,Hours_Studied,Attendance,Previous_Score
0,2,60,40
1,3,65,45
2,4,70,50
3,5,75,55
4,6,80,60
5,7,85,65
6,8,90,70
7,9,95,75
8,10,98,80
9,1,55,35



TARGET:


0    45
1    50
2    55
3    60
4    65
5    70
6    75
7    80
8    85
9    40
Name: Final_Score, dtype: int64

In [5]:
print("DATASET INFORMATION")
df.info()

print("\nSTATISTICAL SUMMARY")
display(df.describe())

print("\nMISSING VALUES")
display(df.isnull().sum())

print("\nFEATURE SHAPE:", X.shape)
print("TARGET SHAPE:", y.shape)

DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Hours_Studied   10 non-null     int64
 1   Attendance      10 non-null     int64
 2   Previous_Score  10 non-null     int64
 3   Final_Score     10 non-null     int64
dtypes: int64(4)
memory usage: 452.0 bytes

STATISTICAL SUMMARY


,Hours_Studied,Attendance,Previous_Score,Final_Score
count,10.00000,10.000000,10.000000,10.000000
mean,5.50000,77.300000,57.500000,62.500000
std,3.02765,14.817782,15.138252,15.138252
min,1.00000,55.000000,35.000000,40.000000
25%,3.25000,66.250000,46.250000,51.250000
50%,5.50000,77.500000,57.500000,62.500000
75%,7.75000,88.750000,68.750000,73.750000
max,10.00000,98.000000,80.000000,85.000000



MISSING VALUES


Hours_Studied     0
Attendance        0
Previous_Score    0
Final_Score       0
dtype: int64


FEATURE SHAPE: (10, 3)
TARGET SHAPE: (10,)


In [6]:
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("TRAINING FEATURES:")
display(X_train)

print("\nTESTING FEATURES:")
display(X_test)

print("\nTRAINING TARGET:")
display(y_train)

print("\nTESTING TARGET:")
display(y_test)

TRAINING FEATURES:


,Hours_Studied,Attendance,Previous_Score
5,7,85,65
0,2,60,40
7,9,95,75
2,4,70,50
9,1,55,35
4,6,80,60
3,5,75,55
6,8,90,70



TESTING FEATURES:


,Hours_Studied,Attendance,Previous_Score
8,10,98,80
1,3,65,45



TRAINING TARGET:


5    70
0    45
7    80
2    55
9    40
4    65
3    60
6    75
Name: Final_Score, dtype: int64


TESTING TARGET:


8    85
1    50
Name: Final_Score, dtype: int64

In [7]:
print("DATASET SPLIT SUMMARY")

print("Original dataset:", len(df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("Training percentage:", len(X_train) / len(df) * 100)
print("Testing percentage:", len(X_test) / len(df) * 100)

DATASET SPLIT SUMMARY
Original dataset: 10
Training records: 8
Testing records: 2
Training percentage: 80.0
Testing percentage: 20.0


## Data Leakage

Data leakage occurs when information from the testing dataset is used
during model training or preprocessing.

This can produce misleading evaluation results.

### Incorrect Approach
Fitting the StandardScaler on the entire dataset before separating
training and testing information.

### Correct Approach
1. Split the dataset first.
2. Fit the scaler only on training features.
3. Transform training features.
4. Transform testing features using the same fitted scaler.

In [8]:
# INCORRECT APPROACH - DATA LEAKAGE

scaler_wrong = StandardScaler()

# Fit scaler using the entire dataset
X_scaled_wrong = scaler_wrong.fit_transform(X)

print("INCORRECT APPROACH - DATA LEAKAGE")

print("\nScaled dataset:")
display(pd.DataFrame(
    X_scaled_wrong,
    columns=X.columns
))

print("\nScaler fitted using the entire dataset.")
print("Testing data information was included.")

INCORRECT APPROACH - DATA LEAKAGE

Scaled dataset:


,Hours_Studied,Attendance,Previous_Score
0,-1.218544,-1.230670,-1.218544
1,-0.870388,-0.874985,-0.870388
2,-0.522233,-0.519300,-0.522233
3,-0.174078,-0.163615,-0.174078
4,0.174078,0.192070,0.174078
5,0.522233,0.547755,0.522233
6,0.870388,0.903440,0.870388
7,1.218544,1.259125,1.218544
8,1.566699,1.472536,1.566699
9,-1.566699,-1.586355,-1.566699



Scaler fitted using the entire dataset.
Testing data information was included.


In [9]:
# CORRECT APPROACH - PREVENT DATA LEAKAGE

scaler = StandardScaler()

# Fit only on training features
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing features using training statistics
X_test_scaled = scaler.transform(X_test)

print("CORRECT APPROACH - NO DATA LEAKAGE")

print("\nScaled Training Features:")
display(pd.DataFrame(
    X_train_scaled,
    columns=X.columns
))

print("\nScaled Testing Features:")
display(pd.DataFrame(
    X_test_scaled,
    columns=X.columns
))

CORRECT APPROACH - NO DATA LEAKAGE

Scaled Training Features:


,Hours_Studied,Attendance,Previous_Score
0,0.664411,0.664411,0.664411
1,-1.233905,-1.233905,-1.233905
2,1.423737,1.423737,1.423737
3,-0.474579,-0.474579,-0.474579
4,-1.613569,-1.613569,-1.613569
5,0.284747,0.284747,0.284747
6,-0.094916,-0.094916,-0.094916
7,1.044074,1.044074,1.044074



Scaled Testing Features:


,Hours_Studied,Attendance,Previous_Score
0,1.803400,1.651535,1.803400
1,-0.854242,-0.854242,-0.854242


In [10]:
print("DATA LEAKAGE PREVENTION SUMMARY")

print("Original dataset:", X.shape)
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

print("\nScaler fitted only on training data.")
print("Testing data was not used during fitting.")

assert X_train_scaled.shape == (8, 3)
assert X_test_scaled.shape == (2, 3)

print("\nAll verification checks passed!")

DATA LEAKAGE PREVENTION SUMMARY
Original dataset: (10, 3)
Training shape: (8, 3)
Testing shape: (2, 3)

Scaler fitted only on training data.
Testing data was not used during fitting.

All verification checks passed!
